# 第312章: テキスト補間とプロンプト変容

## 📋 この章で学ぶこと

- [ ] Classifier-Free Guidance（CFG）の仕組みを条件付き拡散モデルで実装できる
- [ ] 2つのクラス条件の間でプロンプト（条件）を補間して画像を変容させられる
- [ ] CFGスケールが変容品質と多様性に与える影響を実験的に理解できる
- [ ] テキスト直交補間（Prompt-to-Prompt的な発想）の概念を説明できる

## 🎯 前提知識

- ✅ Notebook 310-311（DDIM Inversion、ノイズ空間補間）
- ✅ 拡散モデルの条件付き生成の基礎

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★★☆（上級）
🎓 **カテゴリ**: 理論・実践

---

## 🌟 はじめに

311章ではノイズ空間（xₜ）を補間しました。
この章では**条件ベクトル（プロンプト/クラス）を補間**します。

### アプローチの違い

```
  311章: ノイズ空間補間        本章: 条件補間
  ───────────────────         ──────────────
  xₐ_T ──(α)── x_B_T         cA ──(α)── c_B
      ↓ DDIM Sampling               ↓
      変容画像                   同じxₜから
                             異なる条件でSampling
```

### Classifier-Free Guidance（CFG）

CFGは条件付き生成をさらに強化する技術です：

$$\hat{\epsilon}(x_t, c) = \epsilon(x_t, \varnothing) + w \cdot [\epsilon(x_t, c) - \epsilon(x_t, \varnothing)]$$

- w > 1: 条件をより強く反映（高品質だが多様性低）
- w = 1: 通常の条件付き生成
- w = 0: 無条件生成

### 📝 この章の構成

1. **条件付き拡散モデルの実装** — クラス埋め込み付きDDIM
2. **条件補間** — クラスAとクラスBの間を補間
3. **CFGスケールの実験** — w値が変容に与える影響
4. **Prompt-to-Prompt的発想** — 条件の一部を入れ替える
5. **DDIM Inversion + 条件補間** — 実画像への応用

In [ ]:
# ============================================================
# 環境設定
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)
print(f"Device: {device}")
print("✅ 環境設定完了")

In [ ]:
# ============================================================
# 条件付き拡散モデル (Classifier-Free Guidance対応)
# ============================================================

def get_schedule(n_steps=100, beta_start=1e-4, beta_end=0.02):
    betas = np.linspace(beta_start, beta_end, n_steps)
    alphas = 1.0 - betas
    alpha_bars = np.cumprod(alphas)
    return {
        'betas': torch.tensor(betas, dtype=torch.float32),
        'alphas': torch.tensor(alphas, dtype=torch.float32),
        'alpha_bars': torch.tensor(alpha_bars, dtype=torch.float32),
    }

class ConditionalNoisePredictor(nn.Module):
    """クラス条件付きノイズ予測器 (CFG対応)"""
    def __init__(self, dim=784, n_classes=10, time_emb_dim=32, class_emb_dim=32):
        super().__init__()
        self.null_token = nn.Parameter(torch.zeros(class_emb_dim))  # 無条件用
        self.class_emb = nn.Embedding(n_classes, class_emb_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_emb_dim), nn.SiLU(), nn.Linear(time_emb_dim, time_emb_dim)
        )
        self.net = nn.Sequential(
            nn.Linear(dim + time_emb_dim + class_emb_dim, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, dim),
        )

    def forward(self, x, t, c=None):
        """c=None のとき無条件生成（CFG用）"""
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / 100.0)
        if c is None:
            c_emb = self.null_token.unsqueeze(0).expand(x.shape[0], -1)
        else:
            c_emb = self.class_emb(c)
        x_inp = torch.cat([x, t_emb, c_emb], dim=-1)
        return self.net(x_inp)

    def forward_cfg(self, x, t, c, w=7.5):
        """CFGスケールwでのノイズ予測"""
        eps_uncond = self.forward(x, t, c=None)
        eps_cond = self.forward(x, t, c=c)
        return eps_uncond + w * (eps_cond - eps_uncond)

    def forward_interp(self, x, t, c1, c2, alpha, w=7.5):
        """2つの条件を補間したノイズ予測"""
        emb1 = self.class_emb(c1)
        emb2 = self.class_emb(c2)
        c_interp_emb = (1 - alpha) * emb1 + alpha * emb2
        # 補間した埋め込みでの条件付き予測
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / 100.0)
        eps_interp = self.net(torch.cat([x, t_emb, c_interp_emb], dim=-1))
        eps_uncond = self.forward(x, t, c=None)
        return eps_uncond + w * (eps_interp - eps_uncond)

print("✅ 条件付きモデル定義完了")

In [ ]:
# ============================================================
# 学習 (Classifier-Free Guidance)
# ============================================================

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

schedule = get_schedule(n_steps=100)
alpha_bars = schedule['alpha_bars'].to(device)

cond_model = ConditionalNoisePredictor().to(device)
optimizer = optim.Adam(cond_model.parameters(), lr=2e-4)

p_uncond = 0.15  # 15%の確率で無条件学習（CFGのために必要）

print("条件付き拡散モデル学習中...")
for epoch in range(2):
    cond_model.train()
    total = 0
    for x, y in train_loader:
        x = x.view(-1, 784).to(device) * 2 - 1
        y = y.to(device)
        t = torch.randint(0, 100, (x.shape[0],)).to(device)
        ab = alpha_bars[t].unsqueeze(-1)
        noise = torch.randn_like(x)
        x_noisy = torch.sqrt(ab) * x + torch.sqrt(1 - ab) * noise

        # p_uncond%の確率でクラスをNoneに（無条件学習）
        mask_uncond = torch.rand(x.shape[0], device=device) < p_uncond
        y_inp = y.clone()
        y_inp[mask_uncond] = -1  # ダミー値（forwardでNoneに相当）

        # 学習時は直接埋め込みを使う簡易実装
        t_emb = cond_model.time_mlp(t.float().unsqueeze(-1) / 100.0)
        c_emb = torch.where(
            mask_uncond.unsqueeze(-1),
            cond_model.null_token.unsqueeze(0).expand(x.shape[0], -1),
            cond_model.class_emb(y)
        )
        noise_pred_out = cond_model.net(torch.cat([x_noisy, t_emb, c_emb], dim=-1))
        loss = nn.functional.mse_loss(noise_pred_out, noise)

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1}/15 | Loss: {total/len(train_loader):.6f}")

cond_model.eval()
print("✅ 学習完了")

---

## 1. 条件付き生成の確認

In [ ]:
# ============================================================
# 各数字の条件付き生成
# ============================================================

@torch.no_grad()
def ddim_sample_cfg(model, x_T, class_label, w=7.5, n_steps=5):
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    x = x_T.clone()
    c = torch.full((x.shape[0],), class_label, dtype=torch.long).to(device)
    for i in range(n_steps, 0, -1):
        t_c, t_p = step_idx[i], step_idx[i - 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model.forward_cfg(x, t_t, c, w=w)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_p]) * x0_p + torch.sqrt(1 - ab[t_p]) * eps
    return x

fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for digit in range(10):
    for row, w in enumerate([1.0, 7.5]):
        x_T = torch.randn(1, 784).to(device)
        x_gen = ddim_sample_cfg(cond_model, x_T, digit, w=w)
        img = ((x_gen[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
        axes[row, digit].imshow(img, cmap='gray')
        axes[row, digit].set_title(str(digit), fontsize=12)
        axes[row, digit].axis('off')
    axes[0, 0].set_ylabel('w=1.0', fontsize=11, rotation=0, labelpad=35)
    axes[1, 0].set_ylabel('w=7.5', fontsize=11, rotation=0, labelpad=35)
fig.suptitle('CFG条件付き生成（上: 低w, 下: 高w）', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_312_01_conditional_generation.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 w=7.5のほうが各数字の特徴がよりはっきり")

---

## 2. 条件補間（プロンプト変容）

In [ ]:
# ============================================================
# 条件埋め込みを補間して画像を変容
# ============================================================

@torch.no_grad()
def ddim_sample_interp(model, x_T, c1, c2, alpha, w=5.0, n_steps=5):
    """2つのクラス条件を補間してサンプリング"""
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    x = x_T.clone()
    c1_t = torch.full((x.shape[0],), c1, dtype=torch.long).to(device)
    c2_t = torch.full((x.shape[0],), c2, dtype=torch.long).to(device)
    for i in range(n_steps, 0, -1):
        t_c, t_p = step_idx[i], step_idx[i - 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model.forward_interp(x, t_t, c1_t, c2_t, alpha, w=w)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_p]) * x0_p + torch.sqrt(1 - ab[t_p]) * eps
    return x

n_interp = 9
alphas = np.linspace(0, 1, n_interp)

# 4ペアで条件補間
pairs = [(0, 8), (1, 7), (3, 8), (4, 9)]

fig, axes = plt.subplots(len(pairs), n_interp, figsize=(16, 7))
torch.manual_seed(0)

for row, (ca, cb) in enumerate(pairs):
    x_T = torch.randn(1, 784).to(device)  # 固定ノイズ
    for col, alpha in enumerate(alphas):
        x_gen = ddim_sample_interp(cond_model, x_T, ca, cb, alpha, w=5.0)
        img = ((x_gen[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0: axes[row, col].set_title(f'α={alpha:.2f}', fontsize=9)
    axes[row, 0].set_ylabel(f'{ca}→{cb}', fontsize=12, rotation=0, labelpad=30)

fig.suptitle('条件補間（プロンプト変容）
同じノイズから条件だけを変化させて変容',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_312_02_condition_interpolation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CFGスケール w の影響
# ============================================================

fig, axes = plt.subplots(4, n_interp, figsize=(16, 7))
w_values = [1.0, 3.0, 7.5, 15.0]
ca, cb = 3, 8
torch.manual_seed(42)
x_T_fixed = torch.randn(1, 784).to(device)

for row, w in enumerate(w_values):
    for col, alpha in enumerate(alphas):
        x_gen = ddim_sample_interp(cond_model, x_T_fixed, ca, cb, alpha, w=w)
        img = ((x_gen[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0: axes[row, col].set_title(f'α={alpha:.2f}', fontsize=9)
    axes[row, 0].set_ylabel(f'w={w}', fontsize=11, rotation=0, labelpad=35)

fig.suptitle(f'CFGスケール wの影響（{ca}→{cb}変容）
高wほど条件が強く反映される',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_312_03_cfg_scale_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 低w(1.0): ぼやけた変容  高w(15.0): シャープだが時々不自然")

In [ ]:
# ============================================================
# DDIM Inversion + 条件置換（Prompt-to-Prompt的発想）
# 元の画像を反転したあと、異なる条件でSamplingする
# ============================================================

@torch.no_grad()
def ddim_inversion_cond(model, x_0, class_label, w=1.0, n_steps=5):
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    c = torch.full((x_0.shape[0],), class_label, dtype=torch.long).to(device)
    x = x_0.clone()
    for i in range(n_steps):
        t_c, t_n = step_idx[i], step_idx[i + 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model.forward_cfg(x, t_t, c, w=w)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_n]) * x0_p + torch.sqrt(1 - ab[t_n]) * eps
    return x

test_batch, test_labels = next(iter(test_loader))
test_imgs = test_batch.view(-1, 784).to(device) * 2 - 1

# 数字3の実画像を抽出してInversion後に異なる条件でSampling
idx3 = (test_labels == 3).nonzero()[0].item()
real_3 = test_imgs[idx3:idx3+1]
label_3 = test_labels[idx3].item()

fig, axes = plt.subplots(2, 5, figsize=(14, 5))

# 元画像
img_orig = ((real_3[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
axes[0, 0].imshow(img_orig, cmap='gray')
axes[0, 0].set_title('元の3', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Inversion (条件付き)
noise_3 = ddim_inversion_cond(cond_model, real_3, class_label=3, w=1.0, n_steps=5)

# 異なる条件でSampling
target_labels = [0, 1, 5, 8]
for col, tl in enumerate(target_labels):
    tl_t = torch.full((1,), tl, dtype=torch.long).to(device)
    cond1 = torch.full((1,), 3, dtype=torch.long).to(device)
    x_gen = ddim_sample_interp(cond_model, noise_3, 3, tl, 1.0, w=5.0, n_steps=5)
    img_e = ((x_gen[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[0, col + 1].imshow(img_e, cmap='gray')
    axes[0, col + 1].set_title(f'→{tl}', fontsize=12)
    axes[0, col + 1].axis('off')
axes[0, 0].set_ylabel('Inv+条件置換', fontsize=10, rotation=0, labelpad=55)

# 条件補間（α=0.5）
axes[1, 0].imshow(img_orig, cmap='gray')
axes[1, 0].set_title('元の3', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')
for col, tl in enumerate(target_labels):
    x_gen = ddim_sample_interp(cond_model, noise_3, 3, tl, 0.5, w=5.0, n_steps=5)
    img_e = ((x_gen[0].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[1, col + 1].imshow(img_e, cmap='gray')
    axes[1, col + 1].set_title(f'3+{tl}
α=0.5', fontsize=10)
    axes[1, col + 1].axis('off')
axes[1, 0].set_ylabel('条件補間α=0.5', fontsize=10, rotation=0, labelpad=55)

fig.suptitle('Inversion後の条件変換（Prompt-to-Promptの発想）
上: 完全な条件置換  下: 中間的な条件補間',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_312_04_prompt_to_prompt.png', dpi=150, bbox_inches='tight')
plt.show()

---

## まとめ

### 🎯 学んだこと

- ✓ CFGにより条件付き生成を強化（w > 1でより明確な条件反映）
- ✓ クラス埋め込みを補間することで条件間の滑らかな変容
- ✓ DDIM Inversion後に条件を変えて再生成 = Prompt-to-Promptの基本的な発想

### ⚠️ よくある間違い

❌ 「条件補間と実画像のInversionは同じこと」
✅ 条件補間は「同じノイズ」から異なる条件を使う。Inversionは実画像から始める。両者を組み合わせると強力

### ✅ 学習チェックリスト

- [ ] CFGスケールwの数式を書けるか？
- [ ] 条件補間とノイズ空間補間の違いを説明できるか？

---

## 🎓 自己評価クイズ

### Q1: CFGでw=0のときは何が起きるか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 無条件生成になる（ε_uncond だけが使われる）

w=0の式: ε_uncond + 0*(ε_cond - ε_uncond) = ε_uncond。クラス条件が無視された生成になる。

</details>

---

### Q2: 条件補間（α=0.5）とノイズ補間（α=0.5）を組み合わせるとどのような効果が期待できるか？

<details>
<summary>💡 答えを見る</summary>

**答え**: さらに滑らかで多様な変容が実現できる

ノイズ補間は「どの画像の特性を持つか」を制御し、条件補間は「どのクラスの特徴を反映するか」を制御する。組み合わせることで2つの軸で制御可能になる。

</details>